In [1]:
import pandas as pd
path = "https://raw.githubusercontent.com/ulka29/paper/refs/heads/main/01_data/processed/engineered_feature_dataset.csv"# import Check dataset
df= pd.read_csv(path)
df.head()

,tweet_text,cyberbullying_type,clean_tweet,lemmatized_tweet,sentiment_scores,negative,positive,neutral,compound,word_count,...,second_person_count,exclamation_count,question_count,comma_count,period_count,repeated_exclamation_count,repeated_question_count,repeated_punctuation_count,mention_count,hashtag_count
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr,word katandandre food crapilicious mkr,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000,9.0,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...,aussietv white mkr theblock imacelebrityau tod...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000,14.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes,classy whore red velvet cupcake,"{'neg': 0.32, 'neu': 0.481, 'pos': 0.199, 'com...",0.320,0.199,0.481,-0.4137,9.0,...,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...,meh p thanks head concerned another angry dude...,"{'neg': 0.243, 'neu': 0.607, 'pos': 0.15, 'com...",0.243,0.150,0.607,-0.4854,18.0,...,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,1.0,0.0
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...,isi account pretending kurdish account like is...,"{'neg': 0.129, 'neu': 0.691, 'pos': 0.18, 'com...",0.129,0.180,0.691,0.0258,18.0,...,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,1.0,0.0


In [2]:
print("Class-wise counts:")
for class_name, count in df['cyberbullying_type'].value_counts().items():
    print(f"{class_name}: {count}")

Class-wise counts:
religion: 7963
age: 7959
gender: 7478
ethnicity: 7426
not_cyberbullying: 6435
other_cyberbullying: 5341


In [3]:
print(df.columns.tolist())

['tweet_text', 'cyberbullying_type', 'clean_tweet', 'lemmatized_tweet', 'sentiment_scores', 'negative', 'positive', 'neutral', 'compound', 'word_count', 'char_count', 'unique_word_count', 'profanity_count', 'unique_profanity_count', 'profanity_ratio', 'profanity_presence', 'second_person_count', 'exclamation_count', 'question_count', 'comma_count', 'period_count', 'repeated_exclamation_count', 'repeated_question_count', 'repeated_punctuation_count', 'mention_count', 'hashtag_count']


In [4]:

# ============================================================
# CELL 0: SHARED DATA + FIXED SPLIT + TF-IDF
# ============================================================

import time
import warnings
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# Use the prepared dataframe
df_exp = df.copy()

text_col = "lemmatized_tweet"
target_col = "cyberbullying_type"

sentiment_features = [
    "negative", "positive", "neutral", "compound"
]

behavioral_features = [
    "second_person_count",
    "mention_count",
    "hashtag_count",
    "profanity_count",
    "unique_profanity_count",
    "profanity_ratio",
    "profanity_presence"
]

numeric_features = sentiment_features + behavioral_features

# Validate required columns
required = [text_col, target_col] + numeric_features
missing = [c for c in required if c not in df_exp.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Remove missing targets; preserve all six valid classes
df_exp = df_exp.dropna(subset=[target_col]).copy()
df_exp = df_exp[df_exp[target_col].astype(str).str.strip() != ""]

X_text = df_exp[text_col].fillna("").astype(str)

X_num = (
    df_exp[numeric_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y_raw = df_exp[target_col].astype(str)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
class_names = label_encoder.classes_

print("Classes:", list(class_names))
print("Dataset:", len(y))
print("Class counts:")
print(y_raw.value_counts())

# One fixed 70:30 stratified split
indices = np.arange(len(y))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_text_train = X_text.iloc[train_idx]
X_text_test = X_text.iloc[test_idx]

X_num_train = X_num.iloc[train_idx]
X_num_test = X_num.iloc[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

# Fit vocabulary ONLY on training text
tfidf = TfidfVectorizer(
    analyzer="word",
    max_features=10000,
    ngram_range=(1, 2),
    dtype=np.float32
)

X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)

# Scale numeric features for linear models
scaler = StandardScaler()

X_num_train_scaled = scaler.fit_transform(
    X_num_train
).astype(np.float32)

X_num_test_scaled = scaler.transform(
    X_num_test
).astype(np.float32)

# Sparse matrices for linear models
X_train_linear = hstack(
    [X_tfidf_train, csr_matrix(X_num_train_scaled)],
    format="csr", dtype=np.float32
)

X_test_linear = hstack(
    [X_tfidf_test, csr_matrix(X_num_test_scaled)],
    format="csr", dtype=np.float32
)

# Same feature columns, numeric values left unscaled for trees
X_train_tree = hstack(
    [X_tfidf_train, csr_matrix(
        X_num_train.to_numpy(dtype=np.float32)
    )],
    format="csr", dtype=np.float32
)

X_test_tree = hstack(
    [X_tfidf_test, csr_matrix(
        X_num_test.to_numpy(dtype=np.float32)
    )],
    format="csr", dtype=np.float32
)

# Fixed CV folds for every model's training data
cv = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=42
)

print("\nTrain:", len(y_train), "| Test:", len(y_test))
print("TF-IDF train/test:", X_tfidf_train.shape,
      X_tfidf_test.shape)
print("Linear matrix:", X_train_linear.shape)
print("Tree matrix:", X_train_tree.shape)
print("CV folds: 5")
print("Selection metric: macro-F1")


Classes: ['age', 'ethnicity', 'gender', 'not_cyberbullying', 'other_cyberbullying', 'religion']
Dataset: 42602
Class counts:
cyberbullying_type
religion               7963
age                    7959
gender                 7478
ethnicity              7426
not_cyberbullying      6435
other_cyberbullying    5341
Name: count, dtype: int64

Train: 29821 | Test: 12781
TF-IDF train/test: (29821, 10000) (12781, 10000)
Linear matrix: (29821, 10011)
Tree matrix: (29821, 10011)
CV folds: 5
Selection metric: macro-F1


In [5]:

# ============================================================
# MODEL 1: LOGISTIC REGRESSION
# ============================================================

from sklearn.linear_model import LogisticRegression

start = time.time()

model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    random_state=42
)

param_grid = {
    "C": [0.1, 1.0, 10.0],
    "class_weight": [None]
}

search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    refit=True,
    return_train_score=False
)

search.fit(X_train_linear, y_train)
pred = search.predict(X_test_linear)

print("\nBest parameters:", search.best_params_)
print("Best CV macro-F1:", round(search.best_score_, 4))
print("Accuracy    :", round(accuracy_score(y_test, pred), 4))
print("Macro-F1    :", round(f1_score(
    y_test, pred, average="macro"
), 4))
print("Weighted-F1 :", round(f1_score(
    y_test, pred, average="weighted"
), 4))

print("\nClassification Report\n-------------------------")
print(classification_report(
    y_test, pred, labels=np.arange(len(class_names)),
    target_names=class_names, digits=4, zero_division=0
))

print("Confusion Matrix\n-------------------------")
print(confusion_matrix(y_test, pred))
print(f"Elapsed: {(time.time()-start)/60:.2f} minutes")


Fitting 5 folds for each of 3 candidates, totalling 15 fits

Best parameters: {'C': 1.0, 'class_weight': None}
Best CV macro-F1: 0.8268
Accuracy    : 0.847
Macro-F1    : 0.8293
Weighted-F1 : 0.8489

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age     0.9540    0.9719    0.9629      2388
          ethnicity     0.9754    0.9785    0.9769      2228
             gender     0.9269    0.8306    0.8761      2243
  not_cyberbullying     0.5922    0.6256    0.6084      1931
other_cyberbullying     0.5849    0.6149    0.5995      1602
           religion     0.9538    0.9498    0.9518      2389

           accuracy                         0.8470     12781
          macro avg     0.8312    0.8285    0.8293     12781
       weighted avg     0.8520    0.8470    0.8489     12781

Confusion Matrix
-------------------------
[[2321    2    4   36   25    0]
 [   3 2180    6   12   23    4]
 [   8    7 1863  221  132   12

In [6]:

# ============================================================
# MODEL 2: LINEAR SVM
# ============================================================

from sklearn.svm import LinearSVC

start = time.time()

model = LinearSVC(
    random_state=42,
    max_iter=1000
)

param_grid = {
    "C": [0.1, 1.0, 10.0],
    "class_weight": [None]
}

search = GridSearchCV(
    model, param_grid,
    scoring="f1_macro", cv=cv,
    n_jobs=-1, verbose=2, refit=True
)

search.fit(X_train_linear, y_train)
pred = search.predict(X_test_linear)

print("\nBest parameters:", search.best_params_)
print("Best CV macro-F1:", round(search.best_score_, 4))
print("Accuracy    :", round(accuracy_score(y_test, pred), 4))
print("Macro-F1    :", round(f1_score(
    y_test, pred, average="macro"
), 4))
print("Weighted-F1 :", round(f1_score(
    y_test, pred, average="weighted"
), 4))

print("\nClassification Report\n-------------------------")
print(classification_report(
    y_test, pred, labels=np.arange(len(class_names)),
    target_names=class_names, digits=4, zero_division=0
))

print("Confusion Matrix\n-------------------------")
print(confusion_matrix(y_test, pred))
print(f"Elapsed: {(time.time()-start)/60:.2f} minutes")


Fitting 5 folds for each of 3 candidates, totalling 15 fits

Best parameters: {'C': 0.1, 'class_weight': None}
Best CV macro-F1: 0.8253
Accuracy    : 0.8456
Macro-F1    : 0.8254
Weighted-F1 : 0.8452

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age     0.9388    0.9828    0.9603      2388
          ethnicity     0.9622    0.9834    0.9727      2228
             gender     0.9252    0.8333    0.8768      2243
  not_cyberbullying     0.5992    0.5930    0.5960      1931
other_cyberbullying     0.5877    0.6042    0.5959      1602
           religion     0.9431    0.9577    0.9504      2389

           accuracy                         0.8456     12781
          macro avg     0.8260    0.8257    0.8254     12781
       weighted avg     0.8460    0.8456    0.8452     12781

Confusion Matrix
-------------------------
[[2347    1    2   19   19    0]
 [   3 2191    5    4   20    5]
 [  11   10 1869  218  124   1

In [7]:

# ============================================================
# MODEL 3: SGD CLASSIFIER
# ============================================================

from sklearn.linear_model import SGDClassifier

start = time.time()

model = SGDClassifier(
    loss="log_loss",
    random_state=42,
    max_iter=1000,
    tol=1e-3
)

param_grid = {
    "alpha": [1e-5, 1e-4, 1e-3],
    "penalty": ["l2"],
    "class_weight": [None]
}

search = GridSearchCV(
    model, param_grid,
    scoring="f1_macro", cv=cv,
    n_jobs=-1, verbose=2, refit=True
)

search.fit(X_train_linear, y_train)
pred = search.predict(X_test_linear)

print("\nBest parameters:", search.best_params_)
print("Best CV macro-F1:", round(search.best_score_, 4))
print("Accuracy    :", round(accuracy_score(y_test, pred), 4))
print("Macro-F1    :", round(f1_score(
    y_test, pred, average="macro"
), 4))
print("Weighted-F1 :", round(f1_score(
    y_test, pred, average="weighted"
), 4))

print("\nClassification Report\n-------------------------")
print(classification_report(
    y_test, pred, labels=np.arange(len(class_names)),
    target_names=class_names, digits=4, zero_division=0
))

print("Confusion Matrix\n-------------------------")
print(confusion_matrix(y_test, pred))
print(f"Elapsed: {(time.time()-start)/60:.2f} minutes")


Fitting 5 folds for each of 3 candidates, totalling 15 fits

Best parameters: {'alpha': 1e-05, 'class_weight': None, 'penalty': 'l2'}
Best CV macro-F1: 0.8203
Accuracy    : 0.8391
Macro-F1    : 0.8192
Weighted-F1 : 0.8404

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age     0.9591    0.9619    0.9605      2388
          ethnicity     0.9788    0.9735    0.9761      2228
             gender     0.9074    0.8301    0.8671      2243
  not_cyberbullying     0.5629    0.6701    0.6118      1931
other_cyberbullying     0.5919    0.5206    0.5540      1602
           religion     0.9415    0.9498    0.9456      2389

           accuracy                         0.8391     12781
          macro avg     0.8236    0.8177    0.8192     12781
       weighted avg     0.8443    0.8391    0.8404     12781

Confusion Matrix
-------------------------
[[2297    2    6   60   22    1]
 [   2 2169    4   17   28    8]
 [   5 

In [8]:

# ============================================================
# MODEL 4: RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestClassifier

start = time.time()

model = RandomForestClassifier(
    random_state=42,
    n_jobs=1
)

param_grid = {
    "n_estimators": [200],
    "max_depth": [None, 40],
    "max_features": ["sqrt"],
    "min_samples_leaf": [1, 2],
    "class_weight": [None]
}

search = GridSearchCV(
    model, param_grid,
    scoring="f1_macro", cv=cv,
    n_jobs=-1, verbose=2, refit=True
)

search.fit(X_train_tree, y_train)
pred = search.predict(X_test_tree)

print("\nBest parameters:", search.best_params_)
print("Best CV macro-F1:", round(search.best_score_, 4))
print("Accuracy    :", round(accuracy_score(y_test, pred), 4))
print("Macro-F1    :", round(f1_score(
    y_test, pred, average="macro"
), 4))
print("Weighted-F1 :", round(f1_score(
    y_test, pred, average="weighted"
), 4))

print("\nClassification Report\n-------------------------")
print(classification_report(
    y_test, pred, labels=np.arange(len(class_names)),
    target_names=class_names, digits=4, zero_division=0
))

print("Confusion Matrix\n-------------------------")
print(confusion_matrix(y_test, pred))
print(f"Elapsed: {(time.time()-start)/60:.2f} minutes")


Fitting 5 folds for each of 4 candidates, totalling 20 fits

Best parameters: {'class_weight': None, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 200}
Best CV macro-F1: 0.8331
Accuracy    : 0.8542
Macro-F1    : 0.837
Weighted-F1 : 0.8571

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age     0.9749    0.9765    0.9757      2388
          ethnicity     0.9887    0.9829    0.9858      2228
             gender     0.9380    0.8292    0.8803      2243
  not_cyberbullying     0.5907    0.6779    0.6313      1931
other_cyberbullying     0.5904    0.5930    0.5917      1602
           religion     0.9620    0.9527    0.9573      2389

           accuracy                         0.8542     12781
          macro avg     0.8408    0.8354    0.8370     12781
       weighted avg     0.8622    0.8542    0.8571     12781

Confusion Matrix
-------------------------
[[2332    1    2   2

In [9]:

# ============================================================
# MODEL 5: EXTRA TREES
# ============================================================

from sklearn.ensemble import ExtraTreesClassifier

start = time.time()

model = ExtraTreesClassifier(
    random_state=42,
    n_jobs=1
)

param_grid = {
    "n_estimators": [200],
    "max_depth": [None, 40],
    "max_features": ["sqrt"],
    "min_samples_leaf": [1, 2],
    "class_weight": [None]
}

search = GridSearchCV(
    model, param_grid,
    scoring="f1_macro", cv=cv,
    n_jobs=-1, verbose=2, refit=True
)

search.fit(X_train_tree, y_train)
pred = search.predict(X_test_tree)

print("\nBest parameters:", search.best_params_)
print("Best CV macro-F1:", round(search.best_score_, 4))
print("Accuracy    :", round(accuracy_score(y_test, pred), 4))
print("Macro-F1    :", round(f1_score(
    y_test, pred, average="macro"
), 4))
print("Weighted-F1 :", round(f1_score(
    y_test, pred, average="weighted"
), 4))

print("\nClassification Report\n-------------------------")
print(classification_report(
    y_test, pred, labels=np.arange(len(class_names)),
    target_names=class_names, digits=4, zero_division=0
))

print("Confusion Matrix\n-------------------------")
print(confusion_matrix(y_test, pred))
print(f"Elapsed: {(time.time()-start)/60:.2f} minutes")


Fitting 5 folds for each of 4 candidates, totalling 20 fits

Best parameters: {'class_weight': None, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 200}
Best CV macro-F1: 0.8314
Accuracy    : 0.8517
Macro-F1    : 0.8324
Weighted-F1 : 0.8529

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age     0.9658    0.9807    0.9732      2388
          ethnicity     0.9790    0.9829    0.9810      2228
             gender     0.9280    0.8337    0.8783      2243
  not_cyberbullying     0.5875    0.6712    0.6265      1931
other_cyberbullying     0.6003    0.5605    0.5797      1602
           religion     0.9530    0.9581    0.9555      2389

           accuracy                         0.8517     12781
          macro avg     0.8356    0.8312    0.8324     12781
       weighted avg     0.8561    0.8517    0.8529     12781

Confusion Matrix
-------------------------
[[2342    1    2   

In [10]:

# ============================================================
# MODEL 6: XGBOOST
# ============================================================

from xgboost import XGBClassifier

start = time.time()

param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [4, 8],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8],
    "colsample_bytree": [0.8]
}

def make_xgb(device):
    return XGBClassifier(
        objective="multi:softprob",
        num_class=len(class_names),
        eval_metric="mlogloss",
        tree_method="hist",
        device=device,
        random_state=42,
        n_jobs=1
    )

search = GridSearchCV(
    make_xgb("cuda"),
    param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=1,
    verbose=2,
    refit=True
)

used_device = "cuda"

try:
    search.fit(X_train_tree, y_train)

except Exception as exc:
    print("CUDA run failed:", repr(exc))
    print("Retrying the complete grid on CPU...")

    used_device = "cpu"
    search = GridSearchCV(
        make_xgb("cpu"),
        param_grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=1,
        verbose=2,
        refit=True
    )
    search.fit(X_train_tree, y_train)

pred = search.predict(X_test_tree)

print("\nDevice:", used_device)
print("Best parameters:", search.best_params_)
print("Best CV macro-F1:", round(search.best_score_, 4))
print("Accuracy    :", round(accuracy_score(y_test, pred), 4))
print("Macro-F1    :", round(f1_score(
    y_test, pred, average="macro"
), 4))
print("Weighted-F1 :", round(f1_score(
    y_test, pred, average="weighted"
), 4))

print("\nClassification Report\n-------------------------")
print(classification_report(
    y_test, pred, labels=np.arange(len(class_names)),
    target_names=class_names, digits=4, zero_division=0
))

print("Confusion Matrix\n-------------------------")
print(confusion_matrix(y_test, pred))
print(f"Elapsed: {(time.time()-start)/60:.2f} minutes")


Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=200, subsample=0.8; total time=   5.7s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=200, subsample=0.8; total time=   4.9s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=200, subsample=0.8; total time=   4.9s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=200, subsample=0.8; total time=   5.2s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=200, subsample=0.8; total time=   5.1s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=400, subsample=0.8; total time=   9.4s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=400, subsample=0.8; total time=   8.9s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, n_estimators=400, subsample=0.8; total time=   8.8s
[CV] END col

In [11]:

# ============================================================
# MODEL 7: MLP — OPTIONAL SEPARATE BENCHMARK
# ============================================================

from sklearn.decomposition import TruncatedSVD
from sklearn.neural_network import MLPClassifier

start = time.time()

# Fit dimensionality reduction on training TF-IDF only
svd = TruncatedSVD(n_components=300, random_state=42)

Z_train = svd.fit_transform(X_tfidf_train).astype(np.float32)
Z_test = svd.transform(X_tfidf_test).astype(np.float32)

# Append the same scaled numeric features
X_train_mlp = np.hstack([Z_train, X_num_train_scaled])
X_test_mlp = np.hstack([Z_test, X_num_test_scaled])

model = MLPClassifier(
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)

param_grid = {
    "hidden_layer_sizes": [(128,), (256,)],
    "alpha": [0.0001, 0.001],
    "learning_rate_init": [0.001]
}

search = GridSearchCV(
    model, param_grid,
    scoring="f1_macro", cv=cv,
    n_jobs=1, verbose=2, refit=True
)

search.fit(X_train_mlp, y_train)
pred = search.predict(X_test_mlp)

print("\nSVD dimensions:", 300)
print("Explained variance ratio sum:",
      round(svd.explained_variance_ratio_.sum(), 4))
print("Best parameters:", search.best_params_)
print("Best CV macro-F1:", round(search.best_score_, 4))
print("Accuracy    :", round(accuracy_score(y_test, pred), 4))
print("Macro-F1    :", round(f1_score(
    y_test, pred, average="macro"
), 4))
print("Weighted-F1 :", round(f1_score(
    y_test, pred, average="weighted"
), 4))

print("\nClassification Report\n-------------------------")
print(classification_report(
    y_test, pred, labels=np.arange(len(class_names)),
    target_names=class_names, digits=4, zero_division=0
))

print("Confusion Matrix\n-------------------------")
print(confusion_matrix(y_test, pred))
print(f"Elapsed: {(time.time()-start)/60:.2f} minutes")


Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV] END alpha=0.0001, hidden_layer_sizes=(128,), learning_rate_init=0.001; total time=   8.1s
[CV] END alpha=0.0001, hidden_layer_sizes=(128,), learning_rate_init=0.001; total time=  10.6s
[CV] END alpha=0.0001, hidden_layer_sizes=(128,), learning_rate_init=0.001; total time=   7.9s
[CV] END alpha=0.0001, hidden_layer_sizes=(128,), learning_rate_init=0.001; total time=   6.6s
[CV] END alpha=0.0001, hidden_layer_sizes=(128,), learning_rate_init=0.001; total time=  11.2s
[CV] END alpha=0.0001, hidden_layer_sizes=(256,), learning_rate_init=0.001; total time=  12.2s
[CV] END alpha=0.0001, hidden_layer_sizes=(256,), learning_rate_init=0.001; total time=  10.3s
[CV] END alpha=0.0001, hidden_layer_sizes=(256,), learning_rate_init=0.001; total time=  10.5s
[CV] END alpha=0.0001, hidden_layer_sizes=(256,), learning_rate_init=0.001; total time=  13.7s
[CV] END alpha=0.0001, hidden_layer_sizes=(256,), learning_rate_init=0.001; total tim